In [ ]:
from f5_tts.api import F5TTS
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets, DatasetDict


device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "mansi_v5_Rus"

model_path = f"models/{model_name}/model.pt"
vocab_path = f"models/{model_name}/vocab.txt"

model = F5TTS(ckpt_file=model_path, vocab_file=vocab_path, device=device)
pred_path = 'inference'

ref_audio_path = '../data/tts_dataset/train/wavs/0a7de388-67fb-465b-92b6-99bf0422d118.wav'
ref_text = 'Туня нэ̄мхо̄т ат суйтыгла̄лы, ат на̄ӈкила̄лы, тав сэ̄ръёр.'

In [ ]:
tts_path = "../data/tts_dataset"
tts_val_dataset = load_dataset('csv', data_files=tts_path+'/val/metadata.csv', delimiter='\t', split='train')

stt_path = "../data/stt_dataset"
stt_val_dataset = load_dataset('csv', data_files=stt_path+'/val/metadata.csv', delimiter='\t', split='train')

bonus_path = "../data/bonus_dataset"
bonus_val_dataset = load_dataset('csv', data_files=bonus_path+'/val/metadata.csv', delimiter='\t', split='train')\


dataset = DatasetDict({
    "val": concatenate_datasets([tts_val_dataset, stt_val_dataset, bonus_val_dataset]),
})

dataset = dataset["val"].shuffle(seed=42).select(range(100)).remove_columns('speaker')
dataset

In [ ]:
import os
os.makedirs(pred_path, exist_ok=True)
os.makedirs(f'{pred_path}/wavs', exist_ok=True)

metadata = pd.DataFrame(columns=['model_out', 'reference'])

for sample in dataset:
    model.infer(ref_file=ref_audio_path,
                    ref_text=ref_text,
                    gen_text=sample['text'],
                    file_wave=f'{pred_path}/wavs/{sample['audio_file']}',
                    speed=0.75
                    )
    new_row = pd.DataFrame([{
        'model_out': sample['audio_file'],
        'reference': sample['text']}])
    metadata = pd.concat([metadata, new_row], ignore_index=True)

metadata.to_csv(f'{pred_path}/metadata.csv', index=False, sep="\t")

In [ ]:
!zip -r tts_inference.zip inference